# 🛝 練習場（playground）

ここは **あなたが自由にコードを書く場所** です。教材の演習（`exercises/mX/`）を解いたり、
`src/vla_learn/` の部品をいじって挙動を確かめたりするのに使ってください。

- このノートは**コピーして使ってOK**（例: `m1.ipynb`, `m4_experiment.ipynb` のように章ごとに複製）。
- 出力（画像・テンソルの shape）がその場で見えるので、演習・実験に向いています。
- 起動: リポジトリのルートで `uv sync --extra notebook` → `uv run jupyter lab`
  （カーネルは `.venv` の Python。`import vla_learn` が通ります）。

下のセルを上から実行すると、環境・モデル・学習ループの最小例が一通り動きます。

In [ ]:
# vla_learn を import できるようにする（uv の editable 導入があれば不要。保険として src/ を探す）
import sys, pathlib
try:
    import vla_learn  # editable 導入済みなら通る
except ModuleNotFoundError:
    here = pathlib.Path.cwd()
    for d in [here, *here.parents]:
        if (d / "src" / "vla_learn").exists():
            sys.path.insert(0, str(d / "src")); break
    import vla_learn

import numpy as np
import torch
print("vla_learn", vla_learn.__version__, "| torch", torch.__version__)

## 1. 環境を見る（Vision + Language）\n指示と画像を表示します。`obs` には `image[3,64,64]` / `state[3]` / `instruction` が入っています。

In [ ]:
import matplotlib.pyplot as plt
from vla_learn.envs import Tabletop2DEnv

env = Tabletop2DEnv(seed=0)
obs = env.reset()
print("instruction:", obs["instruction"])
print("state [ax, ay, gripper]:", obs["state"])
print("image shape:", obs["image"].shape)

plt.imshow(np.transpose(obs["image"], (1, 2, 0)))  # [3,H,W] -> [H,W,3]
plt.title(obs["instruction"]); plt.axis("off"); plt.show()

## 2. 小さな VLA を作って forward（Action）\n画像 + 言語 + 状態 → 行動チャンク `[1, 8, 3]` が出ます。

In [ ]:
from vla_learn.envs import all_instruction_strings
from vla_learn.datasets import CharTokenizer
from vla_learn.models import TinyVLA, count_parameters

tok = CharTokenizer.from_corpus(all_instruction_strings())
model = TinyVLA(vocab_size=tok.vocab_size, chunk_len=8)
print("params:", f"{count_parameters(model):,}")

image  = torch.from_numpy(obs["image"])[None]                 # [1, 3, 64, 64]
state  = torch.from_numpy(obs["state"])[None]                 # [1, 3]
tokens = torch.tensor([tok.encode(obs["instruction"])])       # [1, L]
out = model(image, state, tokens)
print("action chunk shape:", tuple(out.shape))               # [1, 8, 3]

## 3. 「1 バッチに過学習できるか」（学習機構の健全性チェック）

教材で繰り返し出てくる鉄則です。モデルと学習ループが正しければ、小さな 1 バッチには必ず過学習できます。
loss が大きく下がれば配線は OK。

In [ ]:
from torch.utils.data import DataLoader
from vla_learn.datasets import generate_episodes, build_normalizers, SyntheticVLADataset
from vla_learn.training.losses import masked_mse

eps = generate_episodes(8, seed=0)
an, sn = build_normalizers(eps)
ds = SyntheticVLADataset(eps, tok, chunk_len=8, action_normalizer=an, state_normalizer=sn)
batch = next(iter(DataLoader(ds, batch_size=16, shuffle=True)))

opt = torch.optim.Adam(model.parameters(), lr=1e-3)
first = None
for i in range(150):
    pred = model(batch["image"], batch["state"], batch["tokens"])
    loss = masked_mse(pred, batch["action"], batch["pad_mask"])
    opt.zero_grad(); loss.backward(); opt.step()
    if first is None:
        first = loss.item()
print(f"overfit loss {first:.3f} -> {loss.item():.3f}")
assert loss.item() < first, "下がっていない＝どこかおかしい（shape/損失/最適化を見直す）"

---
## ✍️ ここから自由に

下のセルから、演習を解いたり実験したりしてください。例:
- `exercises/m1/README.md` の問題をここで解く
- `image_pool="avg"` や `condition_vision=False` で `TinyVLA` を作り、shape や挙動を比べる
- `env.step(action)` をループで回して、自分でロールアウトを書いてみる

ヒントが欲しくなったら対応する `lessons/mX_*.md`、答え合わせは `solutions/mX/` を見ましょう。

In [ ]:
# 自由に書いてください



In [ ]:
# 自由に書いてください

